# Exploration — CKD : comparaison des méthodes d'imputation

Objectif : comparer objectivement deux stratégies d'imputation des valeurs manquantes
(médiane/mode vs KNNImputer) par validation croisée, avant d'écrire le pipeline de
nettoyage final (`src/data_pipeline.py`).

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

pd.set_option("display.width", 120)
RAW_PATH = "../data/raw/kidney_disease.csv"
RANDOM_STATE = 42

## 1. Chargement et corrections validées

Corrections déjà confirmées lors du diagnostic (aucune nouvelle règle introduite ici) :

- `pcv`, `wc`, `rc` : tabs résiduels et `'\t?'` -> conversion en numérique.
- `dm`, `cad`, `classification` : espaces/tabs résiduels sur les modalités texte
  (`'\tno'`, `'\tyes'`, `' yes'`, `'ckd\t'`) -> valeurs normalisées.
- `sc`, `sod`, `pot` : valeurs cliniquement impossibles identifiées au diagnostic
  (créatinine 48.1/76.0, sodium 4.5, potassium 39.0/47.0) -> mises en `NaN` plutôt que
  conservées ou supprimées, pour être traitées comme les autres valeurs manquantes.

In [2]:
df = pd.read_csv(RAW_PATH)

# pcv, wc, rc : tabs / '?' -> NaN, conversion en numérique
for col in ["pcv", "wc", "rc"]:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"?": np.nan, "nan": np.nan})
    df[col] = pd.to_numeric(df[col], errors="coerce")

# dm, cad, classification : espaces/tabs résiduels sur le texte
for col in ["dm", "cad", "classification"]:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan})

# valeurs médicalement impossibles -> NaN
# Seuils alignes sur src/data_pipeline.py (clean_data) : sc>20 mg/dL,
# sod<100 mEq/L, pot>15 mEq/L -- incompatibles avec la vie.
df.loc[df["sc"] > 20, "sc"] = np.nan
df.loc[df["sod"] < 100, "sod"] = np.nan
df.loc[df["pot"] > 15, "pot"] = np.nan

df = df.drop(columns=["id"])
df["target"] = df["classification"].map({"ckd": 1, "notckd": 0})

print(f"Shape après corrections : {df.shape}")
print(f"Valeurs manquantes totales : {df.isna().sum().sum()}")
print(f"Distribution cible : {df['target'].value_counts().to_dict()}")
assert df["target"].isna().sum() == 0, "classification mal normalisée"
df.head()

Shape après corrections : (400, 26)
Valeurs manquantes totales : 1019
Distribution cible : {1: 250, 0: 150}


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,wc,rc,htn,dm,cad,appet,pe,ane,classification,target
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,7800.0,5.2,yes,yes,no,good,no,no,ckd,1
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,6000.0,NaN,no,no,no,good,no,no,ckd,1
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,7500.0,NaN,no,yes,no,poor,no,yes,ckd,1
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,6700.0,3.9,yes,no,no,poor,yes,yes,ckd,1
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,7300.0,4.6,no,no,no,good,no,no,ckd,1


## 2. Comparaison des méthodes d'imputation (validation croisée)

`StratifiedKFold` à 5 folds (dataset de 400 lignes seulement -> on garde la proportion
de classes dans chaque fold). Dans les deux méthodes, l'imputation est **fit sur le
train du fold et appliquée au val du même fold** (via `Pipeline`/`ColumnTransformer`
passés directement à `cross_val_score`, qui refait fit/transform à chaque fold) —
aucune statistique n'est calculée sur l'ensemble des données avant le split.

- **Méthode A** : `SimpleImputer(median)` sur le numérique, `SimpleImputer(most_frequent)`
  sur le catégoriel.
- **Méthode B** : `KNNImputer(n_neighbors=5)` sur le numérique (après standardisation,
  nécessaire pour que la distance euclidienne du KNN soit comparable entre variables),
  `SimpleImputer(most_frequent)` sur le catégoriel (inchangé par rapport à A, pour isoler
  l'effet de la méthode d'imputation numérique).

Le catégoriel est encodé en one-hot (`drop='if_binary'`, toutes les variables
catégorielles de ce dataset sont binaires) après imputation, dans les deux méthodes.

In [3]:
numeric_cols = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot", "hemo", "pcv", "wc", "rc"]
categorical_cols = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane"]

X = df[numeric_cols + categorical_cols]
y = df["target"]

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(drop="if_binary", handle_unknown="ignore")),
])

# Méthode A — Médiane / Mode
numeric_pipeline_A = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
preprocessor_A = ColumnTransformer([
    ("num", numeric_pipeline_A, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])
model_A = Pipeline([
    ("prep", preprocessor_A),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])

# Méthode B — KNNImputer (k=5)
numeric_pipeline_B = Pipeline([
    ("scale", StandardScaler()),   # gère les NaN (ignorés dans le calcul mean/std, préservés en sortie)
    ("impute", KNNImputer(n_neighbors=5)),
])
preprocessor_B = ColumnTransformer([
    ("num", numeric_pipeline_B, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])
model_B = Pipeline([
    ("prep", preprocessor_B),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores_A = cross_val_score(model_A, X, y, cv=cv, scoring="average_precision")
scores_B = cross_val_score(model_B, X, y, cv=cv, scoring="average_precision")

print("Méthode A (Médiane/Mode) — AUC-PR par fold :", np.round(scores_A, 4))
print("Méthode B (KNNImputer)   — AUC-PR par fold :", np.round(scores_B, 4))

Méthode A (Médiane/Mode) — AUC-PR par fold : [1. 1. 1. 1. 1.]
Méthode B (KNNImputer)   — AUC-PR par fold : [1.     1.     0.9982 1.     1.    ]


## 3. Tableau comparatif

In [4]:
comparison = pd.DataFrame({
    "Méthode": ["A — Médiane / Mode", "B — KNNImputer (k=5) / Mode"],
    "AUC-PR moyen (5 folds)": [scores_A.mean(), scores_B.mean()],
    "Écart-type": [scores_A.std(), scores_B.std()],
}).round(4)

winner_idx = comparison["AUC-PR moyen (5 folds)"].idxmax()
print(f"Méthode retenue : {comparison.loc[winner_idx, 'Méthode']}")
comparison

Méthode retenue : A — Médiane / Mode


,Méthode,AUC-PR moyen (5 folds),Écart-type
0,A — Médiane / Mode,1.0000,0.0000
1,B — KNNImputer (k=5) / Mode,0.9996,0.0007


## 4. Synthèse et choix retenu

Les deux méthodes obtiennent une AUC-PR quasi parfaite sur les 5 folds : **A (Médiane/Mode)
= 1.0000 (écart-type 0.0000)** contre **B (KNNImputer, k=5) = 0.9996 (écart-type 0.0007)**.
Ce résultat s'explique par le dataset lui-même : plusieurs variables (`hemo`, `pcv`, `sg`,
`al`, `rc`) sont fortement discriminantes cliniquement entre CKD et non-CKD, ce qui rend la
classification quasi triviale pour un modèle linéaire — quelle que soit la méthode
d'imputation utilisée en amont.

**Méthode retenue : A — Médiane/Mode.** Elle est très légèrement supérieure en moyenne (et
parfaitement stable, écart-type nul), mais surtout elle est nettement plus simple et moins
risquée sur un dataset de seulement 400 lignes : le `KNNImputer` doit calculer des distances
sur des folds d'entraînement réduits à ~320 lignes, ce qui le rend plus sensible au bruit et
plus coûteux à justifier (choix de `k`, sensibilité à l'échelle) pour un gain nul ici.
Le principe de parcimonie (Occam) s'applique directement : à performance égale (voire
légèrement meilleure), la méthode la plus simple à maintenir et à expliquer est préférable
avant même de considérer un futur passage en production.

`src/data_pipeline.py` sera donc écrit avec `SimpleImputer(strategy="median")` pour le
numérique et `SimpleImputer(strategy="most_frequent")` pour le catégoriel, fit sur le train
uniquement et appliqué au test — cohérent avec la méthode validée ici.

---

*Note : les seuils de valeurs medicalement impossibles (`sc>20`, `sod<100`, `pot>15`) ont ete alignes sur ceux de `src/data_pipeline.py` (`clean_data`) pour que ce notebook reste coherent avec le code final. La comparaison a ete re-executee avec ces seuils corriges : les resultats restent inchanges (A = 1.0000, B = 0.9996), confirmant que la conclusion ne dependait pas du seuil exact choisi pour `sc`.*